# Playing around with Molecular docking and scoring

**Authors:**

* [Albert J. Kooistra](https://drug.ku.dk/staff/?pure=en/persons/612712), 2023-2025, University of Copenhagen

* [Jimmy Caroli](https://drug.ku.dk/staff/?pure=en/persons/708879), 2023, University of Copenhagen

This tutorial is constructed as follows:

* **Molecular docking and scoring**
    - Getting to know our target - structure and interactions
    - Redocking our ligand
    - AlphaFold model assessment
    - Large retrospective docking - feature analysis
    - Docking evaluation
    - Optimizing the docking scoring - the knowledge-based way

## Installation and import of libraries and functions

Simply execute the code cells below to get started.

In [ ]:
# Installing missing libraries necessary for processing the docking poses
!pip install -q rdkit prolif==1.1.0
!pip install -q py3Dmol

In [ ]:
# Import packages and libraries
import matplotlib.pyplot as plt
import MDAnalysis as mda
import numpy as np
import os
import pandas as pd
import prolif as plf
import py3Dmol
import re
import seaborn as sns

from ipywidgets import interact,fixed,IntSlider
from rdkit import Chem
from sklearn.metrics import roc_curve, roc_auc_score
from prolif.plotting.network import LigNetwork
from rdkit import Geometry

# Enable molecular viewer output - Google Colab
# from google.colab import output
# output.enable_custom_widget_manager()

# Enable molecular viewer output - Jupyter
# jupyter labextension install  nglview-js-widgets
# jupyter-nbextension enable nglview --py --sys-prefix

In [ ]:
# Disabling warnings (can be tricky)
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# HELPER FUNCTIONS
# No need to read the code or interpret

# Hover functions for the molecular viewer (Javascript code)
hover_func = """
  function(atom,viewer) {
    if (!atom.label) 
      atom.label = viewer.addLabel(atom.resn + " " + atom.resi,
      {position: atom, backgroundColor: 'black', fontColor:'white'});
  }"""

# Unhover function for the molecular viewer (Javascript code)
unhover_func = """function(atom,viewer) {if (atom.label) { viewer.removeLabel(atom.label); delete atom.label; }}"""

# proLIF - redefining HB-acceptors (slightly wider angle)
class HBAcceptor(plf.interactions.HBAcceptor):
   def __init__(self): 
     super().__init__(angles=(120, 180))

# proLIF - redefining HB-donors (slightly wider angle)
class HBDonor(plf.interactions.HBDonor):
   def __init__(self): 
      super().__init__(angles=(120, 180))

# Functions based on TeachOpenCADD - T007
# https://projects.volkamerlab.org/teachopencadd/index.html

def plot_roc_curves_for_models(score_column, data, save_png = False, negative_scores = True):
    """
    Helper function to plot customized roc curve.

    Parameters
    ----------
    score_column: string
        Name of the column with the scores
    data: dataframe
        Dataframe with two columns: <Score> and "Active"
    save_png: bool
        Save image to disk (default = False)

    Returns
    -------
    fig:
        Figure.
    """

    
    if negative_scores:
      data = data.copy()
      data[score_column] = -1 * data[score_column]

    fig, ax = plt.subplots()
    fig.set_dpi(150)

    # Compute ROC plot with False postive rate and True positive rate
    fpr, tpr, thresholds = roc_curve(data["Active"], data[score_column], pos_label=1)

    # Calculate Area under the curve to display on the plot
    auc = roc_auc_score(data["Active"], data[score_column])

    # Plot the computed values
    ax.plot(fpr, tpr, label=(f"{score_column} AUROC = {auc:.2f}"))

    # Custom settings for the plot
    ax.plot([0, 1], [0, 1], "r--")
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title("Receiver Operating Characteristic")
    ax.legend(loc="lower right")

    # Save plot
    if save_png:
        fig.savefig(f"{DATA}/roc_auc", dpi=300, bbox_inches="tight", transparent=True)

    return fig

## Molecular docking and scoring

In the next part we will be digging deeper into molecular docking, but also our target.
Our protein target of interest is ERK2, also known as MK01 or MAPK1, or in full mitogen-activated protein kinase 1.

ERK2 *act as an integration point for multiple biochemical signals, and are involved in a wide variety of cellular processes such as proliferation, differentiation, transcription regulation and development* (source: Wikipedia). Because of its pivotal role, it is a known key player in multiple therapeutic indications, including cancer, heart disease and several syndromes.

In this notebook, we will dig into a structure of ERK2 in complex with ligand *E94* (PDB-code 4FV7). This ligand is a mystery ligand 🪄, with only a structure and no related publications. The ligand has been part of a screening assay of which the results were deposited in the BindingDB. It has a Ki of 27 nM. 

Let's dig deeper into this structure and use it for a docking screen!


### But first we need to collect this data

In [ ]:
# Getting the full data package and extract
# Only do this if we don't already have it!
if not os.path.isfile("Week_5_Monday_dataset.tgz"):
    # Downloading the dataset
    ! wget -q --show-progress "https://www.dropbox.com/s/d5ts9dkf659pkj0/Week_5_Monday_dataset.tgz?dl=1" -O Week_5_Monday_dataset.tgz
    # Extracting the dataset
    ! tar -xvzf Week_5_Monday_dataset.tgz 2>/dev/null

### Checking the data

In the main folder we have different PDB files:
* 1x ERK2 crystal structure (PDB-code 4FV7) protein and ligand
* 1x AlphaFold model
* 1x training_set.txt - the classification active/inactive for our compounds
* 1x The data archive (the .tgz file)
* 1x this notebook

Then we have a docking folder:
* PLANTS - the PLANTS docking program (more later)
* *_plants.conf - configuration files for PLANTS docking
* The protein structure/model and ligand in 
* challenge.smi - the molecular structure of the challenge compounds in SMILES format

And finally, we have pre-docked all compounds into the crystal structure and the AlphaFold model. For each protein structure/model, we have one <code/name>\_docked folder. In each folder you will find:
* training_features.csv.gz - docking scores and features for the compound docking poses
* training_ifp.csv.gz - interaction fingerprints for the compound docking poses

‼️ Note: we obtained 10 docking poses for each compounds, but in a few cases we had multiple isomers of a compound resulting in more than 10 docking poses for the same compound.

## Structure inspection and protein-ligand interactions

First up: getting an idea of the protein and the ligand.

In [ ]:
# Create a py3Dmol viewer
view = py3Dmol.view()

# Read the protein and show it as a gold cartoon
view.addModel(open("klifs_4fv7_protein.pdb","r").read(), format="pdb")
prot = view.getModel()
prot.setStyle({"cartoon" : {"color" : "gold"}})

# Read the ligand file and show it (ligand has PDB-code E94) as sticks with cyan carbon atoms
prot.addMolData(open("klifs_4fv7_ligand.pdb","r").read(), format="pdb")
prot.setStyle({"resn": "E94"}, {"stick":{"colorscheme" : "cyanCarbon"}})

# Add a hover/unhover functions to get some information about the residues and atoms
view.setHoverable({}, True, hover_func, unhover_func)

# Zoom to the protein and show it!
view.zoomTo()
view.show()

The binding site is the most important part for docking. We need to have a good grasp of which residues are forming the binding site and how the ligand interacts with the protein. So let's zoom in!

In [ ]:
# Let's first make the protein cartoon sligthly transparent
view.setStyle({"cartoon": {"color" : "gold", "opacity": 0.6}})

# Select all residues within 5 angstrom of the ligand
selection = {"resn": "E94", "byres": "true", "expand": 5}

# Color the binding site residues in gold
view.addStyle(selection, {"stick" : {"colorscheme": "goldCarbon"}})

# Revert coloring of the ligand back to cyan
view.addStyle({"resn": "E94"}, {"stick": {"colorscheme": "cyanCarbon"}})

# Zoom into the binding site and show it!
view.zoomTo(selection)
view.show()

But wait, we also have generate an AlphaFold model. How does this one compare?\
For this, use the AF_model.pdb and read it in below and give it a distinctive style and color.

‼️ You can also upload and use your own AlphaFold model you've generated during the preparation exercise. To make sure you can compare them properly, make sure to align/superpose the structures to the pdb files here. For this, you can, for example, use PyMol (or biopython if you want to program it!). You can ask the instructors for help with the alignment process. Then add them in the 3D viewer below. 

In [ ]:
# Read the AlphaFold model and add it as a new model with a gray cartoon
### BEGIN SOLUTION
view.addModel(open("AF_model.pdb", "r").read(), format="pdb")
af_mod = view.getModel()
af_mod.setStyle({"cartoon":{"color": "gray", "opacity": 0.6}})


# Get the AF model and add the same reference ligand for easy comparison
af_mod.addMolData(open("klifs_4fv7_ligand.pdb","r").read(), format="pdb")

# Select all residues within 5 angstrom of the ligand of the latest model (-1)
selection = {"resn": "E94", "byres": "true", "expand": 5, "model": -1}

# Color the AF binding site residues in gray
view.addStyle(selection, {"stick" : {"colorscheme": "grayCarbon"}})

### END SOLUTION

# 1. Read and add the AF_model PDB file
view.addModel(open("AF_model.pdb", "r").read(), format="pdb")
af_mod = view.getModel()
af_mod.setStyle({"cartoon":{"color": "gray", "opacity": 0.6}})

# 2. Add the reference ligand from 4fv7
af_mod.addMolData(open("klifs_4fv7_ligand.pdb","r").read(), format="pdb")

# 3. select the residues within 5 angstrom again

selection = {"resn": "E94", "byres": "true", "expand": 5, "model": -1}

# 4. color the sticks wihtin the binding site
view.addStyle(selection, {"stick" : {"colorscheme": "greenCarbon"}})

view.show()

During the lecture, we discussed so-called IFPs - Interaction FingerPrints. Using a package called prolif (abbreviated at plf), we will generate a pandas dataframe with the interactions observed between the ligand and the protein.

In [ ]:
# Calculating the IFP

# Load corrected and protonated PDB we also used for docking
pmol = Chem.MolFromPDBFile("klifs_4fv7_protein.pdb", removeHs=False)
prot = plf.Molecule(pmol)

# Check if our protein is read correctly and has residues
print(f"Our protein has {prot.n_residues} residues")

# Now load our prepared ligand
lmol = Chem.MolFromPDBFile("klifs_4fv7_ligand.pdb", removeHs=False)
ligand = plf.Molecule(lmol)

# And now calculate our interaction fingerprint
fp = plf.Fingerprint()
fp.run_from_iterable([ligand], prot, progress = False)

# Convert to dataframe and show results
df = fp.to_dataframe()
df

### Now let's visualize and interpret the protein-ligand interactions we obtained

In [ ]:
# Get dataframe with atom numbering for depiction
plot_df = fp.to_dataframe(return_atoms=True)

# Create network for visualization from ligand structure and dataframe
net = LigNetwork.from_ifp(plot_df, ligand)

# Display our network
net.display()

You might have noticed that the nitrile group is not correctly depicted - this is because the interpretation of a PDB file. 
PDB files only contain the location of the atoms and not how they are connected, often resulting in a mistake. 


### **Question**: 
What residues are according to you most important for binding of this ligand and why? (note that the plot above is interactive and you can toggle and move all elements).

Now digging deeper into the 3D view

In [ ]:
# Define the colors of the interactions (otherwise gray)
prolif_colors = {"Hydrophobic": "lime", "HBAcceptor": "red", "HBDonor": "blue"}

# Create a py3Dmol viewer and clean view (in case of old models)
view = py3Dmol.view()
view.removeAllModels()

# Read the protein and show it as a gold cartoon
view.addModel(open("klifs_4fv7_protein.pdb","r").read(), format="pdb")
view.setStyle({"cartoon" : {"color" : "gold", "opacity": 0.6}})

# Read the ligand file and show it (ligand has PDB-code E94) as sticks with cyan carbon atoms
view.addModel(open("klifs_4fv7_ligand.pdb","r").read(), format="pdb")
view.setStyle({"resn": "E94"}, {"stick":{"colorscheme" : "cyanCarbon"}})

# Loop over all the interactions in our pandas dataframe that we rotated (T / transformed)
for i, row in plot_df.T.iterrows():
    
    # Collect residue and interaction information
    lresid, presid, interaction = i
    lindex, pindex = row[0]
    
    # Select interacting residue in protein
    pres = prot[presid]

    # Get numeric residue ID
    res_id = re.sub(r"\D", "", presid)

    # Show the residue as sticks
    view.setStyle({"resi": res_id}, {"stick": {"colorscheme": "goldCarbon"}})

    # get coordinates for both points of the interaction
    p1 = ligand.GetConformer().GetAtomPosition(lindex)
    p2 = pres.GetConformer().GetAtomPosition(pindex)

    # Draw and interaction line
    view.addCylinder(
        {
            "start": dict(x=p1.x, y=p1.y, z=p1.z),
            "end": dict(x=p2.x, y=p2.y, z=p2.z),
            "color": prolif_colors.get(interaction, "grey"), 
            "radius": 0.15,
            "dashed": True,
            "fromCap": 1,
            "toCap": 1,
        }
    )

# Zoom in on ligand
view.zoomTo({"resn": "E94"})

## Redocking our ligand

An important step in assessing if a structure is suitable for docking, is the redocking of the molecule that was co-crystallized. Because if that doesn't work out, how can we expect to properly identify new ligands with it?

In the docking folder, there is the PLANTS docking program together with the prepared ligand in a random conformation (MOL2 format to prevent bond errors such as above) and the prepared protein (also MOL2).

In [ ]:
# REDOCKING: redocking the original ligand with PLANTS

# PLANTS is freely available for academic use via http://www.tcd.uni-konstanz.de/research/plants.php
# By using this, you agree with their licensing terms (see website)
# This academic version was obtained from https://github.com/3D-e-Chem/knime-plants

# Go to the docking folder
%cd docking

# Let's make sure PLANTS is executable
!chmod u+x PLANTS

# Delete any previous results (if any)
!rm -rf results 2>/dev/null

# Execute the docking process (and hide the many evaluation messages)
!./PLANTS --mode screen plants.conf 2>&1 |grep -v SIMEVAL

# Go back to our normal working directory
%cd ..

‼️ Hopefully your docking went through ok within a reasonable amount of time. But take into account that this is just docking 1 molecule, docking thousands or even millions can take a very long time.

In [ ]:
# Inspecting the docking results
scores = pd.read_csv("docking/results/ranking.csv")
scores

**Part 1**

Note the many different docking scores. A list from the PLANTS manual is listed below.


* TOTAL_SCORE: scoring function value obtained during docking
* SCORE_RB_PEN: TOTAL_SCORE plus penalty value for each ligand rotatable bond
* SCORE_NORM_HEVATOMS: TOTAL_SCORE divided by number of ligand heavy atoms
* SCORE_NORM_CRT_HEVATOMS: TOTAL_SCORE divided by cubic root of number of ligand heavy atoms
* SCORE_NORM_WEIGHT: TOTAL_SCORE divided by molecular weight of ligand
* SCORE_NORM_CRT_WEIGHT: divided by cubic root of molecular weight of ligand
* SCORE_RB_PEN_NORM_CRT_HEVATOMS: SCORE_RB_PEN divided by
cubic root of number of ligand heavy atoms
* SCORE_NORM_CONTACT: TOTAL SCORE divided by number of protein- ligand contacts
* EVAL: number of scoring function evaluations
* TIME: docking time


### **Question**: Why are there so many docking scores?
(inital guess / group discussion from you, more about this tomorrow)

In [ ]:
# Inspecting the docking features
features = pd.read_csv("docking/results/features.csv")
features

**Part 2**

The features.csv file contains a several different docking scoring elements, i.e. different docking scoring features. A list from the PLANTS manual is listed below.

*	PLPparthbond: PLP hbond score
*	PLPpartsteric: PLP steric contact score
*	PLPpartmetal: PLP metal interaction score
*	PLPpartrepulsive: PLP donor/donor and acceptor/acceptor repulsion score
*	PLPpartburpolar: PLP buried polar atoms score (polar atoms occluded by nonpolar ones)
*	LIG_NUM_CLASH: number of ligand atoms with PLP score greater zero
*	LIG_NUM_CONTACT: number of ligand atoms with attractive PLP score
*	LIG_NUM_NO_CONTACT: number of ligand atoms with zero PLP score
*	CHEMpartmetal: CHEMSCORE metal interaction score
*	CHEMparthbond: CHEMSCORE hbond score
*	DON: number of ligand donor atoms
*	ACC: number of ligand acceptor atoms
*	UNUSED_DON: number of unpaired ligand donors
*	UNUSED_ACC: number of unpaired ligand acceptors
*	CHEMPLP_CLASH: intra-ligand clash score
*	TRIPOS_TORS: intra-ligand torsion score
*	INTRAPROT_CHEMPLP_PLP: intra-protein score (only calculated for flex. side-chains)
*	ATOMS_OUTSIDE_BINDINGSITE: number of ligand atoms outside binding site


### **Question**: If you go through the list, what groups/types of features do you see (more about this tomorrow)?

(inital guess / group discussion from you, more about this tomorrow)

In [ ]:
Finish the function in the code below to show the ligand and the docking pose.

In [ ]:
# Showing the docking poses in the viewer!

# Creating a function that shows the reference ligand as well as a selected docking pose
def pose_viewer(pose_idx):
  # get the selected docking pose
  mol = poses[pose_idx]

  # Create a py3Dmol viewer and clean view (in case of old models)
  dockview = py3Dmol.view()
  dockview.removeAllModels()
  
  ### BEGIN SOLUTION 
  # Read original ligand file and show it
  dockview.addModel(open("klifs_4fv7_ligand.pdb","r").read(), format="pdb")
  orig_ligand = dockview.getModel()
  orig_ligand.setStyle({"stick":{"colorscheme" : "cyanCarbon"}})

  # Show the selected docking pose
  dockview.addModel(mol, format="mol")
  pose = dockview.getModel()
  pose.setStyle({"stick":{"colorscheme" : "magentaCarbon"}})

  dockview.zoomTo()
  dockview.show()
  ### END SOLUTION
  
  # 1. Show original ligand in sticks with one color
  # 2. Show docking pose in sticks with another color
  # 3. Show the result 

# Read our docking poses (MOL2 format)
lig_suppl = plf.mol2_supplier("docking/results/docked_ligands.mol2")

# Process all docking poses and store in poses list
poses = [] 
for mol in lig_suppl:
  poses.append(Chem.MolToMolBlock(mol))

# Create a slider to go through the docking poses
interact(pose_viewer, pose_idx = IntSlider(min=0, max=len(poses)-1))

## Docking into the AlphaFold structure

We just evaluated docking the ligand in its own structure. By doing a cross-docking into another structure, we can check if we can find the correct binding mode also here (because we know the solution, again supervised!). Given we explored AlphaFold modeling, we can check if it performs as well or not.

In [ ]:
# Go to the docking folder
%cd docking

# Delete any previous results (if any)
!rm -rf results_af 2>/dev/null

# Execute the docking process (and hide the many evaluation messages)
!./PLANTS --mode screen af_plants.conf 2>&1 |grep -v SIMEVAL

# Go back to our normal working directory
%cd ..

In [ ]:
# Inspecting the docking results
scores = pd.read_csv("docking/results_af/ranking.csv")
scores

In [ ]:
# Showing the docking poses in the AlphaFold model

### BEGIN SOLUTION
# Read our docking poses (MOL2 format)
lig_suppl = plf.mol2_supplier("docking/results_af/docked_ligands.mol2")

# Process all docking poses 
poses = [] 
for mol in lig_suppl:
  poses.append(Chem.MolToMolBlock(mol))

# Create a slider to go through the docking poses using the pose_viewer function
interact(pose_viewer, pose_idx = IntSlider(min=0, max=len(poses)-1))
### END SOLUTION

# 1. read the docking poses for the AlphaFold model
# 2. process them in a list
# 3. use the function you finished to create the interactive slider with viewer 

### Question: 
What could be the reason(s) for this difference in docking scores and poses?

## Making it real:  analysing a large(r) docking study

We docked a set of nearly 50 thousand compounds into the 4FV7 crystal structure. All these compounds have been experimentally assayed for their inhibitory activity in ERK2.

Since these docking can take quite some time, we prepared the docking and IFP analysis for each of them. You can find the results in the folder 4fv7_docked. These files can be quite large, so keep an eye on your memory usage since we have 10 docking poses for each molecule, resulting in nearly half a million docking scores, poses and IFPs.

Below we will dig into the docking scoring features, but first a small exercise for you.

In [ ]:
# Inspecting the docking results
# PS The files are compressed with gzip to reduce the file size
features = pd.read_csv("4fv7_docked/training_features.csv.gz")
features.head()

### Deeper insights in our features

Before creating our models, we will try to gain some insights into our features and (hopefully) reducing them.

In [ ]:
# We have (too) many docking poses and scores for Noteable
# For simplicity sake, we will now filter down to just the top ranked compounds (conformation 1)
first_pose = features.index.str.endswith('conf_01')
top_df = features[first_pose]

# Change some seaborn settings (size, font, style)
sns.set(rc={'figure.figsize':(16,8)})
sns.set(font_scale=2)
sns.set(style='whitegrid')

# Create boxplot
bp = sns.boxplot(data = top_df)

# Change the orientation of the labels
output = bp.set_xticklabels(bp.get_xticklabels(), rotation=90)

### Adding the active/inactive classification

In [ ]:
# Add the activity classes to the labels and try again
cpd_classes = pd.read_csv("training_set.txt", sep=" ", dtype=str)
cpd_classes['Active'] = cpd_classes['Active'].astype('int')

# Now let's merge the features with the labels, so we know which one is active and which one is not

# 0. make a copy of the dataframe to work in
merged = top_df.copy()

# 1. We need a new row ID with solely the cpd name and remove the other parts
merged.index = [x.split("_")[0] for x in merged.index]

# 2. Add the "Active" column to our dataframe
merged = merged.merge(cpd_classes, left_index=True, right_on="CPD_ID")
merged.head()

### Create boxplot with the two different classes

In [ ]:
# We need to process the dataframe in such a way that seaborn can handle the classes as well as values

# TRICKY - no need to understand this code, but we try to explain it anyway
# 1. We stack the dataframe meaning: convert all columns and rows into a dataframe of <row name>, <column name>, <value>
# More on stacking here: https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.stack.html
df_stacked = merged.drop(["CPD_ID", "Active"], axis=1).stack().reset_index()
# 2. We give meaningful names to the new dataframe columns
df_stacked.columns = ["Active", "Features", "Values"]
# 3. We use the Row ID and replace it with the "Active" classification
labels = merged["Active"].values.tolist()
df_stacked["Active"] = [labels[x] for x in df_stacked["Active"]]


# Now we can use our transformed data to create a more insightful boxplot
bp = sns.boxplot(x="Features", y="Values", hue="Active", data=df_stacked)
output = bp.set_xticklabels(bp.get_xticklabels(), rotation=90)

### QUESTIONS:

* How many active and how many inactive molecules are there in this set?
* Which of the scoring features seem most promising (name 3)
* Perform a min-max normalization of the data (make a copy) and show the boxplot again. Did this change your opinion about the scoring features?
For more information on the min-max normalization, use Google or e.g. https://www.geeksforgeeks.org/data-normalization-with-pandas/
    


### Assessing the cross-correlation between the features

In [ ]:
# Calculating the cross correlation between the different columns
# This is very easy since it's part of pandas!
corr_matrix = merged.corr(method="pearson")
# Rounding the correlation to 1 digit to style the figure
corr_matrix = corr_matrix.round(1)

# Tweak some seaborn settings (size, font, style)
sns.set(rc={'figure.figsize':(16,16)})
sns.set(font_scale=1.5)
sns.set(style=None)

# Draw a correlation heatmap
sns.heatmap(corr_matrix, annot=True)

In [ ]:
### QUESTION:

# Which of the columns are probably redundant / irrelevant?
# Drop these columns and then continue

In [ ]:
### QUESTION:

# Switch from the pearson correlation to the spearman correlation and redraw the correlation heatmap
# How does this change the output? What is the crucial difference between Spearman and Pearson correlation?
# Are there new columns that seem redundant / irrelevant? If so, drop these columns and then continue

### Information Entropy (Shannon entropy)

We already have a good insight now in the different features.
But we can do one more step that might be helpful especially for e.g. analysing the value of specific interaction in the interaction fingerprints.

Information entropy is a measure of the amount of uncertainty or randomness in a set of data, i.e. how _information rich_ is a specific feature? If there is very low variation in the data and it's value is relatively constant, then there's not too much information we can learn from this feature and vice versa.

The higher the entropy value, the higher the information content. With one single value for everything the information entropy is 0.

In [ ]:
# Here we can reuse our dataset we created for the last boxplot (without the Active column)
df_entropy = df_stacked.drop(["Active"], axis=1).copy()

# 1. Calculate the sum of all values per feature
g_sum = df_entropy.groupby("Features")["Values"].transform("sum")

# 2. Divide the individual values by the total sum for that feature
values = df_entropy["Values"]/g_sum

# 3. Calculate the entropy per value as the negative log for each value multiplied by that value
df_entropy["Entropy"] = -(values*np.log(values))

# 4. Calculate the total information entropy per feature by summing all individual entropy values
information_entropy = df_entropy.groupby('Features',as_index=False,sort=False)['Entropy'].sum()
information_entropy

## Retrospective evaluation of the docking results

During a retrospective validation, we already know what the outcome should be (i.e. supervised learning). In this case we know which compounds are active and which ones are inactive experimentally, and this is something we try to capture with our docking scoring as well.

Here we will evaluate selected docking scores and generate Receiver-Operating-Characteristic (ROC) curves to get more insights.


In [ ]:
# Extract a final docking score from the dataframe
ready = merged[["TOTAL_SCORE", "Active"]].copy()

# Check the output
ready.head()

In [ ]:
# Create a ROC plot for the selected score
plot = plot_roc_curves_for_models("TOTAL_SCORE", ready)

### The final BIG steps

Four important tasks that you can implement in the code cells below. These steps will give you insights in the crucial parts of scoring our docking results for this protein.

* 1. Assess ROC and EF1% performance for each of the features and scoring options (after dropping less relevant features and scores)
* 2. Assess ROC and EF1% of the different interactions in the IFP files (4fv7_docked/training_ifp.csv)
* 3. (Optional if you are pressed for time) Assess the performance of the AlphaFold model docking
* 4. Manually generate a score by combining features + interactions => build your own scoring function

In [ ]:
# TASK 1

In [ ]:
# TASK 2

In [ ]:
# TASK 3

In [ ]:
# TASK 4

# DONE 🎉🎉!

# Pandas recap + Molecular docking and scoring

**Authors:**

* [Albert J. Kooistra](https://drug.ku.dk/staff/?pure=en/persons/612712), 2023-2025, University of Copenhagen

* [Jimmy Caroli](https://drug.ku.dk/staff/?pure=en/persons/708879), 2023, University of Copenhagen

This tutorial consist of a two main parts:

* **Getting more familiar with pandas**
    - You will be using pandas to filter and mould your docking results in this and the next notebook


* **Molecular docking and scoring**
    - Getting to know our target - structure and interactions
    - Redocking our ligand
    - AlphaFold model assessment
    - Large retrospective docking - feature analysis
    - Docking evaluation
    - Optimizing the docking scoring - the knowledge-based way

## Installation and import of libraries and functions

Simply execute the code cells below to get started.

In [ ]:
# Installing missing libraries necessary for processing the docking poses
!pip install -q rdkit prolif==1.1.0
!pip install -q py3Dmol

In [ ]:
# Import packages and libraries
import matplotlib.pyplot as plt
import MDAnalysis as mda
import numpy as np
import os
import pandas as pd
import prolif as plf
import py3Dmol
import re
import seaborn as sns

from ipywidgets import interact,fixed,IntSlider
from rdkit import Chem
from sklearn.metrics import roc_curve, roc_auc_score
from prolif.plotting.network import LigNetwork
from rdkit import Geometry

In [ ]:
# Disabling warnings (can be tricky)
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# HELPER FUNCTIONS
# No need to read the code or interpret

# Hover functions for the molecular viewer (Javascript code)
hover_func = """
  function(atom,viewer) {
    if (!atom.label) 
      atom.label = viewer.addLabel(atom.resn + " " + atom.resi,
      {position: atom, backgroundColor: 'black', fontColor:'white'});
  }"""

# Unhover function for the molecular viewer (Javascript code)
unhover_func = """function(atom,viewer) {if (atom.label) { viewer.removeLabel(atom.label); delete atom.label; }}"""

# proLIF - redefining HB-acceptors (slightly wider angle)
class HBAcceptor(plf.interactions.HBAcceptor):
   def __init__(self): 
     super().__init__(angles=(120, 180))

# proLIF - redefining HB-donors (slightly wider angle)
class HBDonor(plf.interactions.HBDonor):
   def __init__(self): 
      super().__init__(angles=(120, 180))

# Function below based on TeachOpenCADD - T007
# https://projects.volkamerlab.org/teachopencadd/index.html
def plot_roc_curves_for_models(score_column, data, save_png = False, negative_scores = True):
    """
    Helper function to plot customized roc curve.

    Parameters
    ----------
    score_column: string
        Name of the column with the scores
    data: dataframe
        Dataframe with two columns: <Score> and "Active"
    save_png: bool
        Save image to disk (default = False)

    Returns
    -------
    fig:
        Figure.
    """

    
    if negative_scores:
      data = data.copy()
      data[score_column] = -1 * data[score_column]

    fig, ax = plt.subplots()
    fig.set_dpi(150)

    # Compute ROC plot with False postive rate and True positive rate
    fpr, tpr, thresholds = roc_curve(data["Active"], data[score_column], pos_label=1)

    # Calculate Area under the curve to display on the plot
    auc = roc_auc_score(data["Active"], data[score_column])

    # Plot the computed values
    ax.plot(fpr, tpr, label=(f"{score_column} AUROC = {auc:.2f}"))

    # Custom settings for the plot
    ax.plot([0, 1], [0, 1], "r--")
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title("Receiver Operating Characteristic")
    ax.legend(loc="lower right")

    # Save plot
    if save_png:
        fig.savefig(f"{DATA}/roc_auc", dpi=300, bbox_inches="tight", transparent=True)

    return fig

## Molecular docking and scoring

In the next part we will be digging deeper into molecular docking, but also our target.
Our protein target of interest is ERK2, also known as MK01 or MAPK1, or in full mitogen-activated protein kinase 1.

ERK2 *act as an integration point for multiple biochemical signals, and are involved in a wide variety of cellular processes such as proliferation, differentiation, transcription regulation and development* (source: Wikipedia). Because of its pivotal role, it is a known key player in multiple therapeutic indications, including cancer, heart disease and several syndromes.

In this notebook, we will dig into a structure of ERK2 in complex with ligand *E94* (PDB-code 4FV7). This ligand is a mystery ligand 🪄, with only a structure and no related publications. The ligand has been part of a screening assay of which the results were deposited in the BindingDB. It has a Ki of 27 nM. 

Let's dig deeper into this structure and use it for a docking screen!


### But first we need to collect this data

In [ ]:
# Getting the full data package and extract
# Only do this if we don't already have it!
if not os.path.isfile("Week_5_Monday_dataset.tgz"):
    # Downloading the dataset
    ! wget -q --show-progress "https://www.dropbox.com/s/d5ts9dkf659pkj0/Week_5_Monday_dataset.tgz?dl=1" -O Week_5_Monday_dataset.tgz
    # Extracting the dataset
    ! tar -xvzf Week_5_Monday_dataset.tgz 2>/dev/null

### Checking the data

In the main folder we have different PDB files:
* 1x ERK2 crystal structure (PDB-code 4FV7) protein and ligand
* 1x AlphaFold model
* 1x training_set.txt - the classification active/inactive for our compounds
* 1x The data archive (the .tgz file)
* 1x this notebook

Then we have a docking folder:
* PLANTS - the PLANTS docking program (more later)
* *_plants.conf - configuration files for PLANTS docking
* The protein structure/model and ligand in 
* challenge.smi - the molecular structure of the challenge compounds in SMILES format

And finally, we have pre-docked all compounds into the crystal structure and the AlphaFold model. For each protein structure/model, we have one <code/name>\_docked folder. In each folder you will find:
* training_features.csv.gz - docking scores and features for the compound docking poses
* training_ifp.csv.gz - interaction fingerprints for the compound docking poses

‼️ Note: we obtained 10 docking poses for each compounds, but in a few cases we had multiple isomers of a compound resulting in more than 10 docking poses for the same compound.

## Structure inspection and protein-ligand interactions

First up: getting an idea of the protein and the ligand.

In [ ]:
# Create a py3Dmol viewer
view = py3Dmol.view()

# Read the protein and show it as a gold cartoon
view.addModel(open("klifs_4fv7_protein.pdb","r").read(), format="pdb")
prot = view.getModel()
prot.setStyle({"cartoon" : {"color" : "gold"}})

# Read the ligand file and show it (ligand has PDB-code E94) as sticks with cyan carbon atoms
prot.addMolData(open("klifs_4fv7_ligand.pdb","r").read(), format="pdb")
prot.setStyle({"resn": "E94"}, {"stick":{"colorscheme" : "cyanCarbon"}})

# Add a hover/unhover functions to get some information about the residues and atoms
view.setHoverable({}, True, hover_func, unhover_func)

# Zoom to the protein and show it!
view.zoomTo()
view.show()

The binding site is the most important part for docking. We need to have a good grasp of which residues are forming the binding site and how the ligand interacts with the protein. So let's zoom in!

In [ ]:
# Let's first make the protein cartoon sligthly transparent
view.setStyle({"cartoon": {"color" : "gold", "opacity": 0.6}})

# Select all residues within 5 angstrom of the ligand
selection = {"resn": "E94", "byres": "true", "expand": 5}

# Color the binding site residues in gold
view.addStyle(selection, {"stick" : {"colorscheme": "goldCarbon"}})

# Revert coloring of the ligand back to cyan
view.addStyle({"resn": "E94"}, {"stick": {"colorscheme": "cyanCarbon"}})

# Zoom into the binding site and show it!
view.zoomTo(selection)
view.show()

But wait, we also have generate an AlphaFold model. How does this one compare?\
For this, use the AF_model.pdb and read it in below and give it a distinctive style and color.

‼️ You can also upload and use your own AlphaFold model you've generated during the preparation exercise. To make sure you can compare them properly, make sure to align/superpose the structures to the pdb files here. For this, you can, for example, use PyMol (or biopython if you want to program it!). You can ask the instructors for help with the alignment process. Then add them in the 3D viewer below. 

In [ ]:
# Read the AlphaFold model and add it as a new model with a gray cartoon
### BEGIN SOLUTION
view.addModel(open("AF_model.pdb", "r").read(), format="pdb")
af_mod = view.getModel()
af_mod.setStyle({"cartoon":{"color": "gray", "opacity": 0.6}})


# Get the AF model and add the same reference ligand for easy comparison
af_mod.addMolData(open("klifs_4fv7_ligand.pdb","r").read(), format="pdb")

# Select all residues within 5 angstrom of the ligand of the latest model (-1)
selection = {"resn": "E94", "byres": "true", "expand": 5, "model": -1}

# Color the AF binding site residues in gray
view.addStyle(selection, {"stick" : {"colorscheme": "grayCarbon"}})

### END SOLUTION

# 1. Read and add the AF_model PDB file
# 2. Add the reference ligand from 4fv7
# 3. select the residues within 5 angstrom again
# 4. color the sticks wihtin the binding site

view.show()

During the lecture, we discussed so-called IFPs - Interaction FingerPrints. Using a package called prolif (abbreviated at plf), we will generate a pandas dataframe with the interactions observed between the ligand and the protein.

In [ ]:
# Calculating the IFP

# Load corrected and protonated PDB we also used for docking
pmol = Chem.MolFromPDBFile("klifs_4fv7_protein.pdb", removeHs=False)
prot = plf.Molecule(pmol)

# Check if our protein is read correctly and has residues
print(f"Our protein has {prot.n_residues} residues")

# Now load our prepared ligand
lmol = Chem.MolFromPDBFile("klifs_4fv7_ligand.pdb", removeHs=False)
ligand = plf.Molecule(lmol)

# And now calculate our interaction fingerprint
fp = plf.Fingerprint()
fp.run_from_iterable([ligand], prot, progress = False)

# Convert to dataframe and show results
df = fp.to_dataframe()
df

### Now let's visualize and interpret the protein-ligand interactions we obtained

In [ ]:
# Get dataframe with atom numbering for depiction
plot_df = fp.to_dataframe(return_atoms=True)

# Create network for visualization from ligand structure and dataframe
net = LigNetwork.from_ifp(plot_df, ligand)

# Display our network
net.display()

You might have noticed that the nitrile group is not correctly depicted - this is because the interpretation of a PDB file. 
PDB files only contain the location of the atoms and not how they are connected, often resulting in a mistake. 


### **Question**: 
What residues are according to you most important for binding of this ligand and why? (note that the plot above is interactive and you can toggle and move all elements).

Now digging deeper into the 3D view

In [ ]:
# Define the colors of the interactions (otherwise gray)
prolif_colors = {"Hydrophobic": "lime", "HBAcceptor": "red", "HBDonor": "blue"}

# Create a py3Dmol viewer and clean view (in case of old models)
view = py3Dmol.view()
view.removeAllModels()

# Read the protein and show it as a gold cartoon
view.addModel(open("klifs_4fv7_protein.pdb","r").read(), format="pdb")
view.setStyle({"cartoon" : {"color" : "gold", "opacity": 0.6}})

# Read the ligand file and show it (ligand has PDB-code E94) as sticks with cyan carbon atoms
view.addModel(open("klifs_4fv7_ligand.pdb","r").read(), format="pdb")
view.setStyle({"resn": "E94"}, {"stick":{"colorscheme" : "cyanCarbon"}})

# Loop over all the interactions in our pandas dataframe that we rotated (T / transformed)
for i, row in plot_df.T.iterrows():
    
    # Collect residue and interaction information
    lresid, presid, interaction = i
    lindex, pindex = row[0]
    
    # Select interacting residue in protein
    pres = prot[presid]

    # Get numeric residue ID
    res_id = re.sub(r"\D", "", presid)

    # Show the residue as sticks
    view.setStyle({"resi": res_id}, {"stick": {"colorscheme": "goldCarbon"}})

    # get coordinates for both points of the interaction
    p1 = ligand.GetConformer().GetAtomPosition(lindex)
    p2 = pres.GetConformer().GetAtomPosition(pindex)

    # Draw and interaction line
    view.addCylinder(
        {
            "start": dict(x=p1.x, y=p1.y, z=p1.z),
            "end": dict(x=p2.x, y=p2.y, z=p2.z),
            "color": prolif_colors.get(interaction, "grey"), 
            "radius": 0.15,
            "dashed": True,
            "fromCap": 1,
            "toCap": 1,
        }
    )

# Zoom in on ligand
view.zoomTo({"resn": "E94"})

## Redocking our ligand

An important step in assessing if a structure is suitable for docking, is the redocking of the molecule that was co-crystallized. Because if that doesn't work out, how can we expect to properly identify new ligands with it?

In the docking folder, there is the PLANTS docking program together with the prepared ligand in a random conformation (MOL2 format to prevent bond errors such as above) and the prepared protein (also MOL2).

In [ ]:
# REDOCKING: redocking the original ligand with PLANTS

# PLANTS is freely available for academic use via http://www.tcd.uni-konstanz.de/research/plants.php
# By using this, you agree with their licensing terms (see website)
# This academic version was obtained from https://github.com/3D-e-Chem/knime-plants

# Go to the docking folder
%cd docking

# Let's make sure PLANTS is executable
!chmod u+x PLANTS

# Delete any previous results (if any)
!rm -rf results 2>/dev/null

# Execute the docking process (and hide the many evaluation messages)
!./PLANTS --mode screen plants.conf 2>&1 |grep -v SIMEVAL

# Go back to our normal working directory
%cd ..

‼️ Hopefully your docking went through ok within a reasonable amount of time. But take into account that this is just docking 1 molecule, docking thousands or even millions can take a very long time.

In [ ]:
# Inspecting the docking results
scores = pd.read_csv("docking/results/ranking.csv")
scores

**Part 1**

Note the many different docking scores. A list from the PLANTS manual is listed below.


* TOTAL_SCORE: scoring function value obtained during docking
* SCORE_RB_PEN: TOTAL_SCORE plus penalty value for each ligand rotatable bond
* SCORE_NORM_HEVATOMS: TOTAL_SCORE divided by number of ligand heavy atoms
* SCORE_NORM_CRT_HEVATOMS: TOTAL_SCORE divided by cubic root of number of ligand heavy atoms
* SCORE_NORM_WEIGHT: TOTAL_SCORE divided by molecular weight of ligand
* SCORE_NORM_CRT_WEIGHT: divided by cubic root of molecular weight of ligand
* SCORE_RB_PEN_NORM_CRT_HEVATOMS: SCORE_RB_PEN divided by
cubic root of number of ligand heavy atoms
* SCORE_NORM_CONTACT: TOTAL SCORE divided by number of protein- ligand contacts
* EVAL: number of scoring function evaluations
* TIME: docking time


### **Question**: Why are there so many docking scores?
(inital guess / group discussion from you, more about this tomorrow)

In [ ]:
# Inspecting the docking features
features = pd.read_csv("docking/results/features.csv")
features

**Part 2**

The features.csv file contains a several different docking scoring elements, i.e. different docking scoring features. A list from the PLANTS manual is listed below.

*	PLPparthbond: PLP hbond score
*	PLPpartsteric: PLP steric contact score
*	PLPpartmetal: PLP metal interaction score
*	PLPpartrepulsive: PLP donor/donor and acceptor/acceptor repulsion score
*	PLPpartburpolar: PLP buried polar atoms score (polar atoms occluded by nonpolar ones)
*	LIG_NUM_CLASH: number of ligand atoms with PLP score greater zero
*	LIG_NUM_CONTACT: number of ligand atoms with attractive PLP score
*	LIG_NUM_NO_CONTACT: number of ligand atoms with zero PLP score
*	CHEMpartmetal: CHEMSCORE metal interaction score
*	CHEMparthbond: CHEMSCORE hbond score
*	DON: number of ligand donor atoms
*	ACC: number of ligand acceptor atoms
*	UNUSED_DON: number of unpaired ligand donors
*	UNUSED_ACC: number of unpaired ligand acceptors
*	CHEMPLP_CLASH: intra-ligand clash score
*	TRIPOS_TORS: intra-ligand torsion score
*	INTRAPROT_CHEMPLP_PLP: intra-protein score (only calculated for flex. side-chains)
*	ATOMS_OUTSIDE_BINDINGSITE: number of ligand atoms outside binding site


### **Question**: If you go through the list, what groups/types of features do you see (more about this tomorrow)?

(inital guess / group discussion from you, more about this tomorrow)

In [ ]:
Finish the function in the code below to show the ligand and the docking pose.

In [ ]:
# Showing the docking poses in the viewer!

# Creating a function that shows the reference ligand as well as a selected docking pose
def pose_viewer(pose_idx):
  # get the selected docking pose
  mol = poses[pose_idx]

  # Create a py3Dmol viewer and clean view (in case of old models)
  dockview = py3Dmol.view()
  dockview.removeAllModels()
  
  ### BEGIN SOLUTION 
  # Read original ligand file and show it
  dockview.addModel(open("klifs_4fv7_ligand.pdb","r").read(), format="pdb")
  orig_ligand = dockview.getModel()
  orig_ligand.setStyle({"stick":{"colorscheme" : "cyanCarbon"}})

  # Show the selected docking pose
  dockview.addModel(mol, format="mol")
  pose = dockview.getModel()
  pose.setStyle({"stick":{"colorscheme" : "magentaCarbon"}})

  dockview.zoomTo()
  dockview.show()
  ### END SOLUTION
  
  # 1. Show original ligand in sticks with one color
  # 2. Show docking pose in sticks with another color
  # 3. Show the result 

# Read our docking poses (MOL2 format)
lig_suppl = plf.mol2_supplier("docking/results/docked_ligands.mol2")

# Process all docking poses and store in poses list
poses = [] 
for mol in lig_suppl:
  poses.append(Chem.MolToMolBlock(mol))

# Create a slider to go through the docking poses
interact(pose_viewer, pose_idx = IntSlider(min=0, max=len(poses)-1))

## Docking into the AlphaFold structure

We just evaluated docking the ligand in its own structure. By doing a cross-docking into another structure, we can check if we can find the correct binding mode also here (because we know the solution, again supervised!). Given we explored AlphaFold modeling, we can check if it performs as well or not.

In [ ]:
# Go to the docking folder
%cd docking

# Delete any previous results (if any)
!rm -rf results_af 2>/dev/null

# Execute the docking process (and hide the many evaluation messages)
!./PLANTS --mode screen af_plants.conf 2>&1 |grep -v SIMEVAL

# Go back to our normal working directory
%cd ..

In [ ]:
# Inspecting the docking results
scores = pd.read_csv("docking/results_af/ranking.csv")
scores

In [ ]:
# Showing the docking poses in the AlphaFold model

### BEGIN SOLUTION
# Read our docking poses (MOL2 format)
lig_suppl = plf.mol2_supplier("docking/results_af/docked_ligands.mol2")

# Process all docking poses 
poses = [] 
for mol in lig_suppl:
  poses.append(Chem.MolToMolBlock(mol))

# Create a slider to go through the docking poses using the pose_viewer function
interact(pose_viewer, pose_idx = IntSlider(min=0, max=len(poses)-1))
### END SOLUTION

# 1. read the docking poses for the AlphaFold model
# 2. process them in a list
# 3. use the function you finished to create the interactive slider with viewer 

### Question: 
What could be the reason(s) for this difference in docking scores and poses?

## Making it real:  analysing a large(r) docking study

We docked a set of nearly 50 thousand compounds into the 4FV7 crystal structure. All these compounds have been experimentally assayed for their inhibitory activity in ERK2.

Since these docking can take quite some time, we prepared the docking and IFP analysis for each of them. You can find the results in the folder 4fv7_docked. These files can be quite large, so keep an eye on your memory usage since we have 10 docking poses for each molecule, resulting in nearly half a million docking scores, poses and IFPs.

Below we will dig into the docking scoring features, but first a small exercise for you.

In [ ]:
# Inspecting the docking results
# PS The files are compressed with gzip to reduce the file size
features = pd.read_csv("4fv7_docked/training_features.csv.gz")
features.head()

### Deeper insights in our features

Before creating our models, we will try to gain some insights into our features and (hopefully) reducing them.

In [ ]:
# We have (too) many docking poses and scores for Noteable
# For simplicity sake, we will now filter down to just the top ranked compounds (conformation 1)
first_pose = features.index.str.endswith('conf_01')
top_df = features[first_pose]

# Change some seaborn settings (size, font, style)
sns.set(rc={'figure.figsize':(16,8)})
sns.set(font_scale=2)
sns.set(style='whitegrid')

# Create boxplot
bp = sns.boxplot(data = top_df)

# Change the orientation of the labels
output = bp.set_xticklabels(bp.get_xticklabels(), rotation=90)

### Adding the active/inactive classification

In [ ]:
# Add the activity classes to the labels and try again
cpd_classes = pd.read_csv("training_set.txt", sep=" ", dtype=str)
cpd_classes['Active'] = cpd_classes['Active'].astype('int')

# Now let's merge the features with the labels, so we know which one is active and which one is not

# 0. make a copy of the dataframe to work in
merged = top_df.copy()

# 1. We need a new row ID with solely the cpd name and remove the other parts
merged.index = [x.split("_")[0] for x in merged.index]

# 2. Add the "Active" column to our dataframe
merged = merged.merge(cpd_classes, left_index=True, right_on="CPD_ID")
merged.head()

### Create boxplot with the two different classes

In [ ]:
# We need to process the dataframe in such a way that seaborn can handle the classes as well as values

# TRICKY - no need to understand this code, but we try to explain it anyway
# 1. We stack the dataframe meaning: convert all columns and rows into a dataframe of <row name>, <column name>, <value>
# More on stacking here: https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.stack.html
df_stacked = merged.drop(["CPD_ID", "Active"], axis=1).stack().reset_index()
# 2. We give meaningful names to the new dataframe columns
df_stacked.columns = ["Active", "Features", "Values"]
# 3. We use the Row ID and replace it with the "Active" classification
labels = merged["Active"].values.tolist()
df_stacked["Active"] = [labels[x] for x in df_stacked["Active"]]


# Now we can use our transformed data to create a more insightful boxplot
bp = sns.boxplot(x="Features", y="Values", hue="Active", data=df_stacked)
output = bp.set_xticklabels(bp.get_xticklabels(), rotation=90)

### QUESTIONS:

* How many active and how many inactive molecules are there in this set?
* Which of the scoring features seem most promising (name 3)
* Perform a min-max normalization of the data (make a copy) and show the boxplot again. Did this change your opinion about the scoring features?
For more information on the min-max normalization, use Google or e.g. https://www.geeksforgeeks.org/data-normalization-with-pandas/
    


### Assessing the cross-correlation between the features

In [ ]:
# Calculating the cross correlation between the different columns
# This is very easy since it's part of pandas!
corr_matrix = merged.corr(method="pearson")
# Rounding the correlation to 1 digit to style the figure
corr_matrix = corr_matrix.round(1)

# Tweak some seaborn settings (size, font, style)
sns.set(rc={'figure.figsize':(16,16)})
sns.set(font_scale=1.5)
sns.set(style=None)

# Draw a correlation heatmap
sns.heatmap(corr_matrix, annot=True)

In [ ]:
### QUESTION:

# Which of the columns are probably redundant / irrelevant?
# Drop these columns and then continue

### Information Entropy (Shannon entropy)

We already have a good insight now in the different features.
But we can do one more step that might be helpful especially for e.g. analysing the value of specific interaction in the interaction fingerprints.

Information entropy is a measure of the amount of uncertainty or randomness in a set of data, i.e. how _information rich_ is a specific feature? If there is very low variation in the data and it's value is relatively constant, then there's not too much information we can learn from this feature and vice versa.

The higher the entropy value, the higher the information content. With one single value for everything the information entropy is 0.

In [ ]:
# Here we can reuse our dataset we created for the last boxplot (without the Active column)
df_entropy = df_stacked.drop(["Active"], axis=1).copy()

# 1. Calculate the sum of all values per feature
g_sum = df_entropy.groupby("Features")["Values"].transform("sum")

# 2. Divide the individual values by the total sum for that feature
values = df_entropy["Values"]/g_sum

# 3. Calculate the entropy per value as the negative log for each value multiplied by that value
df_entropy["Entropy"] = -(values*np.log(values))

# 4. Calculate the total information entropy per feature by summing all individual entropy values
information_entropy = df_entropy.groupby('Features',as_index=False,sort=False)['Entropy'].sum()
information_entropy

## Retrospective evaluation of the docking results

During a retrospective validation, we already know what the outcome should be (i.e. supervised learning). In this case we know which compounds are active and which ones are inactive experimentally, and this is something we try to capture with our docking scoring as well.

Here we will evaluate selected docking scores and generate Receiver-Operating-Characteristic (ROC) curves to get more insights.


In [ ]:
# Extract a final docking score from the dataframe
ready = merged[["TOTAL_SCORE", "Active"]].copy()

# Check the output
ready.head()

In [ ]:
# Create a ROC plot for the selected score
plot = plot_roc_curves_for_models("TOTAL_SCORE", ready)

### The final BIG steps

Four important tasks that you can implement in the code cells below. These steps will give you insights in the crucial parts of scoring our docking results for this protein.

* 1. Assess ROC and EF1% performance for each of the features and scoring options (after dropping less relevant features and scores)
* 2. Assess ROC and EF1% of the different interactions in the IFP files (4fv7_docked/training_ifp.csv)
* 3. (Optional if you are pressed for time) Assess the perofrmance of the AlphaFold model docking
* 4. Manually generate a score by combining features + interactions => build your own scoring function

In [ ]:
# TASK 1

In [ ]:
# TASK 2

In [ ]:
# TASK 3

In [ ]:
# TASK 4

# FINALLY:

In [ ]:
import time, random
from IPython.display import clear_output, display, Markdown

emojis = ['🎉','🥳','✨','🎓','🚀','🌟','🎈']
for i in range(20):
    clear_output(wait=True)
    frame = " ".join(random.choices(emojis, k=40))
    display(Markdown(f"<div style='font-size:30px;text-align:center'>{frame}</div>"))
    time.sleep(0.25)

clear_output(wait=True)
display(Markdown("<h1 style='text-align:center'>🎓 Congratulations — ULLA AIinDD — Molecular Docking Monday completed! 🎉</h1>"))